# 10 — Paper Figures

Generates all publication-ready figures using final results.

**Figures produced:**

| File | Function | Description |
|---|---|---|
| `occlusion_performance_curve.{png,pdf}` | `plot_occlusion_performance_curve` | AP by difficulty tier per model variant |
| `aug_sweep.{png,pdf}` | `plot_augmentation_strength_sweep` | p vs AP_easy/mod/hard/mAP@0.5 |
| `ablation_bar_chart.{png,pdf}` | `plot_ablation_bar_chart` | ΔAP_hard when each component removed |
| `results_table.{csv,tex}` | `plot_results_table` | Full LaTeX + CSV results table |
| `depth_map_comparison_*.{png,pdf}` | `plot_depth_map_comparison` | RGB / depth / occluded depth / mask |

**Prerequisites:**
- Notebooks `01–08` must have completed (all results in `results/tables/val_results_all_models.csv`)
- Notebook `09` must have completed (test results in `results/tables/test_results_FINAL.csv`)
- Aug sweep results in `results/tables/aug_sweep_results.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd

from src.plotting import (
    plot_occlusion_performance_curve,
    plot_augmentation_strength_sweep,
    plot_ablation_bar_chart,
    plot_results_table,
    plot_depth_map_comparison,
)

ROOT = Path('..').resolve()
TABLES_DIR  = ROOT / 'results' / 'tables'
FIGURES_DIR = ROOT / 'results' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Loading results...')

# Val results (all models)
val_csv = TABLES_DIR / 'val_results_all_models.csv'
if not val_csv.exists():
    raise FileNotFoundError(f'Missing: {val_csv}. Run notebooks 01–08 first.')
val_df = pd.read_csv(val_csv)
print(f'Val results: {len(val_df)} model variants')

# Test results
test_csv = TABLES_DIR / 'test_results_FINAL.csv'
if not test_csv.exists():
    raise FileNotFoundError(f'Missing: {test_csv}. Run notebook 09 first.')
test_df = pd.read_csv(test_csv)
print(f'Test results: {len(test_df)} rows')

# Aug sweep results
sweep_csv = TABLES_DIR / 'aug_sweep_results.csv'
if not sweep_csv.exists():
    raise FileNotFoundError(f'Missing: {sweep_csv}. Run notebook 02 first.')
sweep_df = pd.read_csv(sweep_csv)
optimal_p_file = ROOT / 'results' / 'optimal_aug_p.txt'
optimal_p = float(optimal_p_file.read_text().strip()) if optimal_p_file.exists() else 0.4
print(f'Aug sweep: {len(sweep_df)} p values. Optimal p={optimal_p}')

# Ablation results
abl_csv = TABLES_DIR / 'ablation_results.csv'
if not abl_csv.exists():
    raise FileNotFoundError(f'Missing: {abl_csv}. Run notebook 08 first.')
abl_df = pd.read_csv(abl_csv)
M8_AP_HARD = float(val_df.loc[val_df['model_id'] == 'M8', 'AP_hard'].values[0])
print(f'Ablation: {len(abl_df)} variants. M8 AP_hard reference: {M8_AP_HARD:.2f}')

In [ ]:
# ── Figure 1: Occlusion Performance Curve ─────────────────────────────────────
# Lines: M0, M2, M4, M6, M7, M8 (solid) + M0_hard (dashed)
# Annotated ΔAP mechanism text boxes for each transition

MAIN_MODELS = ['M0', 'M2', 'M4', 'M6', 'M7', 'M8']
perf_data = []
for model_id in MAIN_MODELS:
    row = val_df[val_df['model_id'] == model_id]
    if len(row) > 0:
        r = row.iloc[0]
        perf_data.append({'model_id': model_id, 'AP_easy': r['AP_easy'],
                          'AP_mod': r['AP_mod'], 'AP_hard': r['AP_hard'],
                          'linestyle': 'solid'})

m0h_row = val_df[val_df['model_id'] == 'M0_hard']
if len(m0h_row) > 0:
    r = m0h_row.iloc[0]
    perf_data.append({'model_id': 'M0_hard', 'AP_easy': r['AP_easy'],
                      'AP_mod': r['AP_mod'], 'AP_hard': r['AP_hard'],
                      'linestyle': 'dashed'})

plot_occlusion_performance_curve(
    model_data=perf_data,
    save_dir=FIGURES_DIR,
)
print(f'Saved: {FIGURES_DIR}/occlusion_performance_curve.png')

In [ ]:
# ── Figure 2: Augmentation Strength Sweep ────────────────────────────────────
# AP_easy, AP_mod, AP_hard (primary axis) + mAP@0.5 (secondary axis)

plot_augmentation_strength_sweep(
    sweep_data=sweep_df.to_dict('records'),
    optimal_p=optimal_p,
    save_dir=FIGURES_DIR,
    selection_note=(
        f'Selected p={optimal_p}: highest AP_hard without degrading '
        f'AP_easy by more than 2 points'
    ),
)
print(f'Saved: {FIGURES_DIR}/aug_sweep.png')

In [ ]:
# ── Figure 3: Ablation Bar Chart ──────────────────────────────────────────────
# Horizontal bars sorted by drop magnitude; colour-coded by category

plot_ablation_bar_chart(
    ablation_data=abl_df.to_dict('records'),
    m8_ap_hard=M8_AP_HARD,
    save_dir=FIGURES_DIR,
)
print(f'Saved: {FIGURES_DIR}/ablation_bar_chart.png')

In [ ]:
# ── Figure 4: Results Table (LaTeX + CSV) ────────────────────────────────────
# Combines val results (M0–M8) + test results; best per column bolded

# Build combined table: val AP + test metrics for final model
DISPLAY_MODELS = ['M0', 'M0_hard', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
table_data = []
for model_id in DISPLAY_MODELS:
    row = val_df[val_df['model_id'] == model_id]
    if len(row) == 0:
        continue
    r = row.iloc[0]
    entry = {
        'Model': model_id,
        'AP_easy': r['AP_easy'],
        'AP_mod': r['AP_mod'],
        'AP_hard': r['AP_hard'],
        'ORS': None,
        'FPS': None,
        'Notes': r.get('description', ''),
    }
    # Add test metrics for M8
    if model_id == 'M8' and len(test_df) > 0:
        t = test_df.iloc[0]
        entry['ORS'] = t.get('ORS')
        entry['FPS'] = t.get('FPS_mean')
    table_data.append(entry)

plot_results_table(
    table_data=table_data,
    save_dir=TABLES_DIR,
)
print(f'Saved: {TABLES_DIR}/results_table.csv and .tex')

In [ ]:
# ── Figure 5: Depth Map Comparison (20 examples) ─────────────────────────────
# 4-panel per example: RGB | clean depth | occluded depth | mask overlay

from pathlib import Path
import numpy as np
import torch
from PIL import Image
from src.augmentation import LabelAwareCutout

DATA_ROOT = ROOT / 'data'
kitti_img_dir   = DATA_ROOT / 'kitti' / 'data_object_image_2' / 'training' / 'image_2'
kitti_depth_dir = DATA_ROOT / 'kitti' / 'depth_hybrid' / 'training' / 'image_2'

if kitti_img_dir.exists() and kitti_depth_dir.exists():
    depth_files = sorted(kitti_depth_dir.glob('*.npy'))[:20]
    examples = []
    for df in depth_files:
        img_path = kitti_img_dir / (df.stem + '.png')
        conf_path = df.parent / (df.stem + '_conf.npy')
        if not img_path.exists():
            continue

        img_pil  = Image.open(img_path).resize((640, 640))
        img_arr  = np.array(img_pil).astype(np.float32) / 255.0
        depth    = np.load(df)
        conf     = np.load(conf_path) if conf_path.exists() else np.zeros_like(depth)

        # Apply augmentation to create occluded version
        import torchvision.transforms.functional as TF
        img_t  = torch.from_numpy(img_arr).permute(2, 0, 1)
        depth_t = torch.from_numpy(depth).unsqueeze(0)
        conf_t  = torch.from_numpy(conf).unsqueeze(0)
        dmask_t = torch.ones(1, 640, 640, dtype=torch.bool)
        boxes   = torch.tensor([[0.1, 0.1, 0.5, 0.5]])
        occ_lvls = [0]

        cutout = LabelAwareCutout(p=1.0)
        result = cutout(img_t, depth_t, conf_t, dmask_t, boxes, occ_lvls)

        examples.append({
            'image_id': df.stem,
            'rgb': img_arr,
            'depth_clean': depth,
            'depth_occluded': result.depth[0].numpy(),
            'mask': result.mask_applied.numpy(),
        })

    plot_depth_map_comparison(
        examples=examples,
        save_dir=FIGURES_DIR,
    )
    print(f'Saved: {FIGURES_DIR}/depth_map_comparison_*.png')
else:
    print(f'[SKIP] KITTI images/depths not found — skipping depth map comparison figures.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import os

print('\nAll paper figures generated:')
for f in sorted(FIGURES_DIR.glob('*.png')) + sorted(FIGURES_DIR.glob('*.pdf')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:<50} {size_kb:>6} KB')

print('\nTables:')
for f in sorted(TABLES_DIR.glob('*.csv')) + sorted(TABLES_DIR.glob('*.tex')):
    print(f'  {f.name}')

print('\nPaper figures complete.')